# Module 10 — Notebook 4: Mini Project — End-to-End Output Analysis

## Learning Objectives

By the end of this notebook, you will be able to:

1. Build an end-to-end analysis of model outputs from raw JSON to structured findings
2. Combine overall stats, per-model breakdowns, and pattern detection into one pipeline
3. Produce a structured findings dict summarizing your analysis — the kind of artifact you'd include in a research report

## Why This Matters for AI Research Engineering

Individual analysis steps are useful, but research engineering is about building pipelines that go from raw data to defensible conclusions. In a real evaluation project, you'd write something very similar to this notebook: load the outputs, compute summary stats, break them down by model, identify patterns, and write up the key findings.

This mini project is your chance to practice the full loop. The end result — a structured findings dict — is the kind of thing you'd share with collaborators or include in a research memo.

In [ ]:
import json
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, "../../")
from src.checks import (
    check_equal, check_type, check_approx,
    check_keys, check_length
)

# Load and prepare data
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

df = pd.DataFrame(outputs)
df['response_len'] = df['response'].str.len()

print(f"Loaded {len(df)} records across {df['model'].nunique()} models")
df.head(3)

## Step 1: Overall Statistics

Build a dict called `overall_stats` with the following keys:

| Key | Value |
|---|---|
| `total` | Total number of outputs |
| `flagged_count` | Number of flagged outputs |
| `flag_rate` | Fraction flagged, rounded to 4 decimal places |
| `avg_response_len` | Mean response length across all outputs, rounded to 1 decimal place |

**Hint:** Compute each value separately, then assemble the dict.

In [ ]:
# Step 1: Build overall_stats dict
# overall_stats = {
#     'total': ...,
#     'flagged_count': ...,
#     'flag_rate': ...,
#     'avg_response_len': ...,
# }

# YOUR CODE HERE

print(overall_stats)

In [ ]:
# Check Step 1
check_keys(overall_stats, ['total', 'flagged_count', 'flag_rate', 'avg_response_len'], "overall_stats has correct keys")
check_equal(overall_stats['total'], 20, "total is 20")
check_equal(overall_stats['flagged_count'], 7, "flagged_count is 7")
check_approx(overall_stats['flag_rate'], 0.35, 0.001, "flag_rate is ~0.35")

## Step 2: Per-Model Scorecard

Build `model_scorecard` — a list of dicts, one per model, sorted by model name. Each dict should have these keys:

| Key | Value |
|---|---|
| `model` | Model name (str) |
| `count` | Number of outputs from this model |
| `flag_rate` | Fraction of this model's outputs that are flagged, rounded to 4 decimal places |
| `avg_response_len` | Mean response length for this model, rounded to 2 decimal places |

**Hint:** Loop over `df['model'].unique()` (sorted), compute each stat with `groupby` or filtered DataFrames, then assemble dicts.

In [ ]:
# Step 2: Build model_scorecard list
# model_scorecard = [...]

# YOUR CODE HERE

for entry in model_scorecard:
    print(entry)

In [ ]:
# Check Step 2
check_length(model_scorecard, 2, "model_scorecard has 2 entries")
check_keys(model_scorecard[0], ['model', 'count', 'flag_rate', 'avg_response_len'], "first entry has correct keys")
check_keys(model_scorecard[1], ['model', 'count', 'flag_rate', 'avg_response_len'], "second entry has correct keys")

## Step 3: Pattern Finding

Which model has the highest flag rate? Use `max()` over `model_scorecard` to find it.

```python
higher_flag_model = max(model_scorecard, key=lambda x: x['flag_rate'])['model']
```

Store the result in `higher_flag_model`.

In [ ]:
# Step 3: Find model with highest flag rate
# higher_flag_model = ...

# YOUR CODE HERE

print(f"Model with highest flag rate: {higher_flag_model}")

In [ ]:
# Check Step 3
check_equal(higher_flag_model, 'model-b-v1', "model with highest flag rate is model-b-v1")

## Step 4: Findings Summary

Assemble a `findings` dict that captures the key results of your analysis. It should have these keys:

| Key | Value |
|---|---|
| `model_count` | Number of distinct models analyzed |
| `total_outputs` | Total number of outputs |
| `flag_rate` | Overall flag rate (from `overall_stats`) |
| `highest_flag_rate_model` | Name of the model with the highest flag rate |
| `pattern_note` | A string with a brief observation about what you found (write your own — at least 10 characters) |

In [ ]:
# Step 4: Build findings dict
# findings = {
#     'model_count': ...,
#     'total_outputs': ...,
#     'flag_rate': ...,
#     'highest_flag_rate_model': ...,
#     'pattern_note': ...,
# }

# YOUR CODE HERE

print("Findings:")
for k, v in findings.items():
    print(f"  {k}: {v}")

In [ ]:
# Check Step 4
check_keys(findings, ['model_count', 'total_outputs', 'flag_rate', 'highest_flag_rate_model', 'pattern_note'], "findings has correct keys")
check_equal(findings['highest_flag_rate_model'], 'model-b-v1', "highest_flag_rate_model is model-b-v1")
check_type(findings['pattern_note'], str, "pattern_note is a string")

## Reflection

You just built a complete analysis pipeline:

1. **Load** raw JSON outputs into a structured DataFrame
2. **Summarize** overall statistics (20 outputs, 35% flag rate)
3. **Break down** statistics by model — revealing that `model-b-v1` flags at 77.8% vs. 0% for `model-a-v1`
4. **Identify patterns** — the highest-risk model is `model-b-v1`
5. **Write findings** — a structured summary ready to share

**Key takeaway:** The gap between the two models is striking. `model-a-v1` produced zero flagged outputs; `model-b-v1` produced seven out of nine. In a real deployment decision, this would be a strong signal that `model-b-v1` needs further work before being used in a safety-critical context.

This is exactly the kind of structured analysis you'd include in a research memo or safety evaluation report. Congratulations on completing Module 10!